> **Open in VS Code:** From your cloned `s26-06643` repository, run in the terminal:
> ```bash
> code sse/optional/uv/uv.ipynb
> ```
> [View source on GitHub](https://github.com/jkitchin/s26-06643/blob/main/sse/optional/uv/uv.ipynb)

# uv: Modern Python Package Management

Python package management has historically been slow and fragmented. You need `pip` to install packages, `virtualenv` or `venv` to create environments, `pip-tools` to lock dependencies, and `pyenv` to manage Python versions. [uv](https://docs.astral.sh/uv/) replaces all of these with a single, fast tool. Built in Rust by [Astral](https://astral.sh/) (the same team behind [Ruff](https://docs.astral.sh/ruff/)), uv is 10-100x faster than pip and provides a unified workflow for Python development.

## Section 1: Why uv?

If you have used `pip install` on a project with many dependencies, you know it can be painfully slow. pip resolves dependencies in Python, downloads packages sequentially, and rebuilds its index on every run. uv does all of this in Rust with aggressive caching and parallel downloads.

**What uv replaces:**

| Traditional Tool | Purpose | uv Equivalent |
|-----------------|---------|---------------|
| `pip` | Install packages | `uv pip install` |
| `pip-tools` | Lock dependencies | `uv pip compile` |
| `virtualenv` / `venv` | Create environments | `uv venv` |
| `pyenv` | Manage Python versions | `uv python` |

Key advantages:
- **Speed**: 10-100x faster than pip for installs and dependency resolution
- **Single tool**: One binary replaces four or five separate tools
- **Drop-in compatible**: `uv pip install` works exactly like `pip install` — same flags, same behavior
- **Built in Rust**: By Astral, the same team that built Ruff (the fast Python linter)
- **Cross-platform**: Works on macOS, Linux, and Windows

Installing uv:

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Or with pip (if you already have Python)
pip install uv

# Or with Homebrew
brew install uv
```

In [ ]:
# Check that uv is installed and see its version
!uv --version

## Section 2: Virtual Environments

### Why virtual environments matter

A virtual environment is an isolated Python installation. Without one, every project shares the same global packages. This leads to version conflicts:

- Project A needs `numpy==1.24` but Project B needs `numpy==2.0`
- Installing a package for one project breaks another
- You cannot reproduce your environment on a different machine

Virtual environments solve this by giving each project its own set of installed packages. **You should always work in a virtual environment.** There is no good reason to install packages globally.

### Creating a virtual environment

```bash
# Create a virtual environment in .venv (the default)
uv venv

# Or specify a name
uv venv my-env
```

This creates a `.venv` directory in your current folder. It is much faster than `python -m venv .venv`.

### Activating and deactivating

```bash
# macOS / Linux
source .venv/bin/activate

# Windows
.venv\Scripts\activate

# Deactivate (any platform)
deactivate
```

When a virtual environment is active, your shell prompt will show `(.venv)` and `python` will point to the environment's Python, not your system Python.

### Why you should ALWAYS work in a virtual environment

- **Isolation**: Each project has its own dependencies, no conflicts
- **Reproducibility**: You can freeze your environment and recreate it on another machine
- **Safety**: You will not accidentally break system Python or other projects
- **Cleanliness**: Deleting the `.venv` directory removes everything cleanly

In [ ]:
# Create a virtual environment with uv
!uv venv /tmp/demo-venv

In [ ]:
# Verify that the environment has its own Python
!ls /tmp/demo-venv/bin/python*

### Exercise: Create a venv, install a package, and verify isolation

1. Create a new virtual environment with `uv venv /tmp/isolation-test`
2. Install `requests` into it: `uv pip install requests --python /tmp/isolation-test/bin/python`
3. Verify that `requests` is available inside the environment: `/tmp/isolation-test/bin/python -c "import requests; print(requests.__version__)"`
4. Verify that your system Python does NOT have this installation affected
5. Clean up by removing the directory: `rm -rf /tmp/isolation-test`

## Section 3: Installing Packages

uv provides a drop-in replacement for pip. If you know `pip install`, you already know `uv pip install`.

### Basic installs

```bash
# Install a package
uv pip install numpy

# Install a specific version
uv pip install numpy==2.0.0

# Install multiple packages
uv pip install numpy pandas matplotlib
```

### Editable installs (for development)

When you are developing a package, you want changes to your source code to take effect immediately without reinstalling:

```bash
# Install your project in editable/development mode
uv pip install -e .

# With optional dependencies
uv pip install -e ".[dev,test]"
```

### From requirements files

```bash
# Install from requirements.txt
uv pip install -r requirements.txt
```

### Locking dependencies

Pinning exact versions ensures reproducibility. `uv pip compile` resolves all dependencies and writes pinned versions:

```bash
# Generate a locked requirements.txt from pyproject.toml
uv pip compile pyproject.toml -o requirements.txt

# Or from a requirements.in file
uv pip compile requirements.in -o requirements.txt
```

### Speed comparison

Try this yourself to see the difference:

```bash
# pip (slow)
time pip install scikit-learn

# uv (fast)
time uv pip install scikit-learn
```

On a warm cache, uv typically finishes in under a second where pip takes 10-30 seconds.

In [ ]:
# Install a package with uv into our demo environment
!uv pip install numpy --python /tmp/demo-venv/bin/python

### Exercise: Install a package in a fresh venv

1. Create a fresh virtual environment: `uv venv /tmp/install-test`
2. Install `numpy` and `matplotlib` into it using `uv pip install`
3. List installed packages: `uv pip list --python /tmp/install-test/bin/python`
4. Notice how many transitive dependencies were installed alongside matplotlib
5. Clean up: `rm -rf /tmp/install-test`

## Section 4: Managing Python Versions

uv can also manage Python installations, replacing tools like `pyenv`.

### Installing Python versions

```bash
# Install a specific Python version
uv python install 3.12

# Install multiple versions
uv python install 3.11 3.12 3.13
```

### Listing available Python versions

```bash
# See installed and available Python versions
uv python list
```

### Creating a venv with a specific Python version

```bash
# Create a venv using Python 3.11
uv venv --python 3.11

# Create a venv using Python 3.12
uv venv --python 3.12 .venv-312
```

This is especially useful when you need to test your code against multiple Python versions, or when a project requires a specific version.

In [ ]:
# List available Python versions
!uv python list

### Exercise: Create venvs with different Python versions

1. Install Python 3.11 and 3.12 (if not already installed): `uv python install 3.11 3.12`
2. Create a venv with Python 3.11: `uv venv /tmp/py311-env --python 3.11`
3. Create a venv with Python 3.12: `uv venv /tmp/py312-env --python 3.12`
4. Verify each environment uses the correct Python version:
   - `/tmp/py311-env/bin/python --version`
   - `/tmp/py312-env/bin/python --version`
5. Clean up: `rm -rf /tmp/py311-env /tmp/py312-env`

## Section 5: Running Scripts and Tools

uv can run Python scripts and tools without manually managing environments.

### Running scripts

`uv run` executes a script, automatically resolving and installing any dependencies declared in the script:

```bash
# Run a script in the current project environment
uv run script.py

# Run with additional dependencies
uv run --with requests script.py
```

You can even declare dependencies inline in your script using a special comment format:

```python
# /// script
# dependencies = ["requests", "rich"]
# ///

import requests
from rich import print

response = requests.get("https://api.github.com")
print(response.json())
```

Then just `uv run script.py` and uv handles the rest.

### Installing CLI tools globally

```bash
# Install a CLI tool (available globally)
uv tool install ruff
uv tool install black

# Now you can use them directly
ruff check .
```

### Running tools without installing

```bash
# Run a tool without permanent installation
uv tool run pytest
uv tool run black --check .

# Shorthand: uvx (alias for uv tool run)
uvx pytest
uvx ruff check .
```

This is like `npx` in the Node.js world. The tool is downloaded, run, and then the temporary environment is cached for future use.

In [ ]:
# Run a tool without installing it permanently
!uvx ruff version

### Exercise: Run pytest via uv without installing it permanently

1. Create a simple test file:
   ```python
   # /tmp/test_example.py
   def test_addition():
       assert 1 + 1 == 2
   
   def test_string():
       assert "hello".upper() == "HELLO"
   ```
2. Run the tests with `uvx pytest /tmp/test_example.py`
3. Notice that pytest was never installed into your current environment
4. Verify: `pip show pytest` should show it is not installed in your main environment

## Section 6: uv in Projects

uv has its own project management commands that go beyond pip replacement. These manage your entire project lifecycle.

### Initializing a project

```bash
# Create a new project
uv init my-project
cd my-project
```

This creates a `pyproject.toml`, a `README.md`, and a basic project structure.

### Adding dependencies

```bash
# Add a dependency (updates pyproject.toml and uv.lock)
uv add numpy
uv add pandas matplotlib

# Add a development dependency
uv add --dev pytest ruff

# Remove a dependency
uv remove pandas
```

`uv add` does three things:
1. Adds the package to `pyproject.toml`
2. Resolves all dependencies
3. Updates `uv.lock` with pinned versions

### The lock file

`uv.lock` is a cross-platform lock file that records the exact version of every dependency (including transitive dependencies). This ensures that everyone on your team gets the same versions:

```bash
# Regenerate the lock file
uv lock

# Install from the lock file (exact versions)
uv sync
```

**Always commit `uv.lock` to version control.** This is how you guarantee reproducibility.

### Integration with pyproject.toml

uv reads and writes the standard `[project]` table in `pyproject.toml`:

```toml
[project]
name = "my-project"
version = "0.1.0"
description = "My scientific project"
requires-python = ">=3.11"
dependencies = [
    "numpy>=2.0",
    "matplotlib>=3.8",
]

[dependency-groups]
dev = [
    "pytest>=8.0",
    "ruff>=0.4",
]
```

This is the same standard format used by pip, build tools, and other package managers. Your project is not locked into uv.

In [ ]:
# Initialize a new project in a temporary directory
!cd /tmp && uv init demo-project && cat /tmp/demo-project/pyproject.toml

### Exercise: Convert a pip-based project to use uv

Suppose you have a project that currently uses pip with a `requirements.txt`:

```
numpy>=1.24
pandas>=2.0
matplotlib>=3.7
scipy>=1.11
pytest>=7.0
```

Convert it to use uv:

1. Initialize a new uv project: `uv init /tmp/converted-project`
2. Add the runtime dependencies: `cd /tmp/converted-project && uv add numpy pandas matplotlib scipy`
3. Add dev dependencies: `uv add --dev pytest`
4. Inspect the generated `pyproject.toml` and `uv.lock`
5. Run `uv sync` to verify everything installs correctly
6. Clean up: `rm -rf /tmp/converted-project`

## Tips and Tricks

- **Use `uv pip install` as a drop-in replacement for `pip install`**: The command-line interface is identical, so you can switch without learning new flags.
- **Always create a `.venv` in your project root**: `uv venv` with no arguments does this by default. Most editors (VS Code, PyCharm) auto-detect `.venv` directories.
- **Use `uvx` to try tools without commitment**: `uvx black --check .` or `uvx ruff check .` lets you evaluate tools without polluting your environment.
- **Commit `uv.lock` to version control**: This file ensures every collaborator and CI server uses exactly the same package versions.
- **Use `uv add` instead of manually editing `pyproject.toml`**: It resolves dependencies and updates the lock file in one step, preventing version conflicts.
- **Use inline script dependencies for standalone scripts**: The `# /// script` comment block lets `uv run` auto-install dependencies without a requirements file.
- **Prompt AI with: "Set up this project with uv instead of pip"**: AI agents know uv well and can generate the full `uv init`, `uv add`, and `pyproject.toml` workflow.
- **Use `uv python install` to manage Python versions**: You no longer need pyenv or conda just to get a specific Python version. `uv python install 3.12` handles it.

## Foundational Concepts to Commit to Memory

- **uv is a single tool that replaces pip, pip-tools, virtualenv, and pyenv** -- it handles package installation, dependency locking, environment creation, and Python version management.
- **`uv venv` creates a virtual environment** and is faster than `python -m venv`. Always work in a virtual environment.
- **`uv pip install` is a drop-in replacement for `pip install`** -- same flags, same behavior, 10-100x faster.
- **`uv pip compile` locks dependencies** by resolving all transitive dependencies and writing pinned versions to a requirements file.
- **`uv python install 3.12` installs a Python version** without needing pyenv, conda, or system package managers.
- **`uv run script.py` runs a script** with automatic dependency resolution, including inline script dependencies.
- **`uvx` (alias for `uv tool run`) runs CLI tools without installing them** -- like `npx` in the JavaScript world.
- **`uv add` modifies `pyproject.toml` and updates `uv.lock`** in one step -- always use it instead of hand-editing dependency lists.

## Knowing vs. Doing: Reflection

Package management is not glamorous, but it is the foundation that every Python project rests on. When `pip install` takes 30 seconds and you are iterating on a CI pipeline that installs dependencies on every push, those seconds turn into hours over the life of a project. When a collaborator cannot reproduce your results because `pip install -r requirements.txt` resolves to different versions on their machine, the debugging session costs far more than the minutes it would have taken to set up a lock file.

uv solves these problems not by introducing new concepts but by making the existing workflow faster and more reliable. You still create virtual environments, you still install packages, you still pin dependencies. You just do it with a tool that respects your time. The conceptual model is the same -- isolation through virtual environments, reproducibility through lock files, convenience through a unified CLI.

The important thing is not memorizing every uv subcommand. It is understanding why virtual environments exist, why dependency locking matters, and why speed in tooling translates to better engineering habits. When creating a new environment takes under a second, you are more likely to do it. When installing dependencies is instant, you are more likely to test in a clean environment. Good tools do not just save time -- they lower the friction that keeps you from doing things the right way.

## Additional Resources

- [uv Documentation](https://docs.astral.sh/uv/) -- official documentation for uv, covering installation, commands, and configuration
- [Ruff Documentation](https://docs.astral.sh/ruff/) -- the fast Python linter and formatter from Astral, the same team behind uv
- [Python venv Documentation](https://docs.python.org/3/library/venv.html) -- the standard library module that uv replaces for virtual environment creation